In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm
import re
import sys
from pathlib import Path
import glob
import subprocess

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing_local import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities_local import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


In [5]:
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'
local_path = "local-files"

###

EVENT_NAME = "202411_TropicalStorm_Sara"
product = "sentinel1"

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

keys = [x.replace(f"drcs_activations/{EVENT_NAME}/{product}/", "") for x in get_all_s3_keys(s3_client, BUCKET, f"drcs_activations/{EVENT_NAME}/{product}", ".tif")] if s3_client else []

keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files


['S1A_IW_20241104T113101_DVR_RTC20_G_gpufed_A0BC_WM.tif',
 'S1A_IW_20241104T113101_DVR_RTC20_G_gpufed_A0BC_rgb.tif',
 'S1A_IW_20241104T113126_DVR_RTC20_G_gpufed_5784_WM.tif',
 'S1A_IW_20241104T113126_DVR_RTC20_G_gpufed_5784_rgb.tif',
 'S1A_IW_20241104T235642_DVR_RTC20_G_gpufed_0608_WM.tif',
 'S1A_IW_20241104T235642_DVR_RTC20_G_gpufed_0608_rgb.tif',
 'S1A_IW_20241104T235709_DVR_RTC20_G_gpufed_A74D_WM.tif',
 'S1A_IW_20241104T235709_DVR_RTC20_G_gpufed_A74D_rgb.tif',
 'S1A_IW_20241106T111504_DVR_RTC20_G_gpufed_DB44_WM.tif',
 'S1A_IW_20241106T111504_DVR_RTC20_G_gpufed_DB44_rgb.tif',
 'S1A_IW_20241106T111531_DVR_RTC20_G_gpufed_2D9E_WM.tif',
 'S1A_IW_20241106T111531_DVR_RTC20_G_gpufed_2D9E_rgb.tif',
 'S1A_IW_20241111T112235_DVR_RTC20_G_gpufed_096E_WM.tif',
 'S1A_IW_20241111T112235_DVR_RTC20_G_gpufed_096E_rgb.tif',
 'S1A_IW_20241111T112300_DVR_RTC20_G_gpufed_6BEC_WM.tif',
 'S1A_IW_20241111T112300_DVR_RTC20_G_gpufed_6BEC_rgb.tif',
 'S1A_IW_20241111T112325_DVR_RTC20_G_gpufed_A1DF_WM.tif',
 'S1A_

In [7]:
# Download the desired files to a local directory with a known path
local_file_dir = os.path.abspath(f"./{local_path}")
if not os.path.exists(local_file_dir):
    os.mkdir(local_file_dir)
for key in keys:
    subprocess.run([
        "aws",
        "s3",
        "cp",
        f"s3://nasa-disasters/drcs_activations/{EVENT_NAME}/{product}/{key}",
        local_file_dir], check = True)

download: s3://nasa-disasters/drcs_activations/202411_TropicalStorm_Sara/sentinel1/S1A_IW_20241104T113101_DVR_RTC20_G_gpufed_A0BC_WM.tif to local-files/S1A_IW_20241104T113101_DVR_RTC20_G_gpufed_A0BC_WM.tif
download: s3://nasa-disasters/drcs_activations/202411_TropicalStorm_Sara/sentinel1/S1A_IW_20241104T113101_DVR_RTC20_G_gpufed_A0BC_rgb.tif to local-files/S1A_IW_20241104T113101_DVR_RTC20_G_gpufed_A0BC_rgb.tif
download: s3://nasa-disasters/drcs_activations/202411_TropicalStorm_Sara/sentinel1/S1A_IW_20241104T113126_DVR_RTC20_G_gpufed_5784_WM.tif to local-files/S1A_IW_20241104T113126_DVR_RTC20_G_gpufed_5784_WM.tif
download: s3://nasa-disasters/drcs_activations/202411_TropicalStorm_Sara/sentinel1/S1A_IW_20241104T113126_DVR_RTC20_G_gpufed_5784_rgb.tif to local-files/S1A_IW_20241104T113126_DVR_RTC20_G_gpufed_5784_rgb.tif
download: s3://nasa-disasters/drcs_activations/202411_TropicalStorm_Sara/sentinel1/S1A_IW_20241104T235642_DVR_RTC20_G_gpufed_0608_WM.tif to local-files/S1A_IW_20241104T2356

In [8]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

In [9]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [10]:
def make_regex_dict(keys, regexes, products):
    ret = {}
    for i in range(len(products)):
        matches = []
        for key in keys:
            filename = key.split("/")[-1]
            match = re.search(regexes[i], filename)
            if match is not None:
                matches.append(key)
        if matches != []:
            ret[products[i]] = matches
    return ret

In [11]:
def create_cog_filename(filename, event):
    if re.search(r".*_rgb.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split("_")
        date = datetime.strptime(sname[2], "%Y%m%dT%H%M%S")
        new_dt_format = date.strftime("%Y-%m-%dT%H:%M:%SZ")
        cog_filename = f"{event}_{sname[0]}_{sname[1]}_{sname[3]}_{sname[4]}_{sname[5]}_{sname[6]}_{sname[7]}_{sname[8]}_{new_dt_format}.tif"

    elif re.search(r".*_WM.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split("_")
        date = datetime.strptime(sname[2], "%Y%m%dT%H%M%S")
        new_dt_format = date.strftime("%Y-%m-%dT%H:%M:%SZ")
        cog_filename = f"{event}_{sname[0]}_{sname[1]}_{sname[3]}_{sname[4]}_{sname[5]}_{sname[6]}_{sname[7]}_{sname[8]}_{new_dt_format}.tif"

    else:
        print(f"{filename} not caught by regexes!")
        return None
    
    return cog_filename

In [12]:
local_keys = [x for x in glob.glob(f"{local_path}/*") if x.endswith(".tif")]

local_keys

['local-files/S1A_IW_20241116T113101_DVR_RTC20_G_gpufed_400A_WM.tif',
 'local-files/S1A_IW_20241104T113101_DVR_RTC20_G_gpufed_A0BC_WM.tif',
 'local-files/S1A_IW_20241116T113101_DVR_RTC20_G_gpufed_400A_rgb.tif',
 'local-files/S1A_IW_20241104T113101_DVR_RTC20_G_gpufed_A0BC_rgb.tif',
 'local-files/S1A_IW_20241116T113125_DVR_RTC20_G_gpufed_76FF_WM.tif',
 'local-files/S1A_IW_20241104T113126_DVR_RTC20_G_gpufed_5784_WM.tif',
 'local-files/S1A_IW_20241116T113125_DVR_RTC20_G_gpufed_76FF_rgb.tif',
 'local-files/S1A_IW_20241104T113126_DVR_RTC20_G_gpufed_5784_rgb.tif',
 'local-files/S1A_IW_20241116T235641_DVR_RTC20_G_gpufed_5FB6_WM.tif',
 'local-files/S1A_IW_20241104T235642_DVR_RTC20_G_gpufed_0608_WM.tif',
 'local-files/S1A_IW_20241116T235641_DVR_RTC20_G_gpufed_5FB6_rgb.tif',
 'local-files/S1A_IW_20241104T235642_DVR_RTC20_G_gpufed_0608_rgb.tif',
 'local-files/S1A_IW_20241116T235709_DVR_RTC20_G_gpufed_7B93_rgb.tif',
 'local-files/S1A_IW_20241104T235709_DVR_RTC20_G_gpufed_A74D_WM.tif',
 'local-files

In [13]:
reg_keys = make_regex_dict(local_keys, [r".*_rgb.tif", r".*_WM.tif"], ["RGB/subdaily", "HydroSAR_WM"])

In [14]:
print(reg_keys)
for k, v in reg_keys.items():
    for filename in v:
        print(create_cog_filename(filename, EVENT_NAME))

{'RGB/subdaily': ['local-files/S1A_IW_20241116T113101_DVR_RTC20_G_gpufed_400A_rgb.tif', 'local-files/S1A_IW_20241104T113101_DVR_RTC20_G_gpufed_A0BC_rgb.tif', 'local-files/S1A_IW_20241116T113125_DVR_RTC20_G_gpufed_76FF_rgb.tif', 'local-files/S1A_IW_20241104T113126_DVR_RTC20_G_gpufed_5784_rgb.tif', 'local-files/S1A_IW_20241116T235641_DVR_RTC20_G_gpufed_5FB6_rgb.tif', 'local-files/S1A_IW_20241104T235642_DVR_RTC20_G_gpufed_0608_rgb.tif', 'local-files/S1A_IW_20241116T235709_DVR_RTC20_G_gpufed_7B93_rgb.tif', 'local-files/S1A_IW_20241104T235709_DVR_RTC20_G_gpufed_A74D_rgb.tif', 'local-files/S1A_IW_20241118T111504_DVR_RTC20_G_gpufed_8ABE_rgb.tif', 'local-files/S1A_IW_20241106T111504_DVR_RTC20_G_gpufed_DB44_rgb.tif', 'local-files/S1A_IW_20241118T111531_DVR_RTC20_G_gpufed_01E0_rgb.tif', 'local-files/S1A_IW_20241106T111531_DVR_RTC20_G_gpufed_2D9E_rgb.tif', 'local-files/S1A_IW_20241123T112259_DVR_RTC20_G_gpufed_114B_rgb.tif', 'local-files/S1A_IW_20241111T112235_DVR_RTC20_G_gpufed_096E_rgb.tif', 'l

In [15]:
def simple_process_files(file_list, rename_func, target_dir, event):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    print("Testing filenams:")
    for filename in file_list:
        print(f"  {rename_func(filename, event)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        #"raw_data_bucket": BUCKET,
        #"raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{event}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=file_list,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=event,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

In [16]:
if not os.path.exists(os.path.abspath("./output")):
    os.mkdir(os.path.abspath("./output"))
if not os.path.exists(os.path.abspath("./reproj")):
    os.mkdir(os.path.abspath("./reproj"))
for k, v in reg_keys.items():
    results = simple_process_files(file_list = v, rename_func = create_cog_filename, target_dir = f"Sentinel-1/{k}", event = EVENT_NAME)

Testing filenams:
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_400A_rgb_2024-11-16T11:31:01Z.tif
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_A0BC_rgb_2024-11-04T11:31:01Z.tif
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_76FF_rgb_2024-11-16T11:31:25Z.tif
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_5784_rgb_2024-11-04T11:31:26Z.tif
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_5FB6_rgb_2024-11-16T23:56:41Z.tif
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_0608_rgb_2024-11-04T23:56:42Z.tif
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_7B93_rgb_2024-11-16T23:57:09Z.tif
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_A74D_rgb_2024-11-04T23:57:09Z.tif
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_8ABE_rgb_2024-11-18T11:15:04Z.tif
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_DB44_rgb_2024-11-06T11:15:04Z.tif
  202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_01E0_rgb_2024-11-18T11:15:31Z.tif
  202411_Tropi

Band 1:  47%|████▋     | 85/180 [00:03<00:04, 22.48chunks/s]


   [MEMORY] High usage: 634.5 MB, forcing cleanup...


Band 1:  53%|█████▎    | 96/180 [00:03<00:03, 22.44chunks/s]


   [MEMORY] High usage: 672.1 MB, forcing cleanup...


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=999951/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpc1xyqa2m_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpvp4cxwxc.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/HydroSAR_WM/202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_0C2F_WM_2024-11-23T23:49:03Z.tif
   [MEMORY] Final: 1424.5 MB (Change: -60.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202411_TropicalStorm_Sara_S1A_IW_DVR_RTC20_G_gpufed_0C2F_WM_2024-11-23T23:49:03Z.tif

✅ Batch processing complete: 23 files processed
📁 COGs saved locally to: output/202411_TropicalStorm_Sara

📊 BATCH PROCESSING SUMMARY
Total files processed: 23
Successful: 23
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-29T23:41:08.872966


In [17]:
subprocess.run(["rm", "-r", f"{local_path}"], check = True)
subprocess.run(["rm", "-r", os.path.abspath("./output")], check = True)
subprocess.run(["rm", "-r", os.path.abspath("./reproj")], check = True)

CompletedProcess(args=['rm', '-r', '/home/jovyan/conversion_scripts/convert-files-and-move/2024/reproj'], returncode=0)